In [42]:
# -*- coding: utf-8 -*-
"""polisy_core: shared configuration, IO and keys for the POLISY data pipeline.

Design rules enforced here:
  * nothing large is ever read into pandas; DuckDB reads the raw files and returns
    aggregates, Polars handles medium frames, pandas only holds final tables
  * every raw file is converted once to Parquet in DATA/canonical and read from there
  * every canonical table has exactly one grain, declared in GRAINS below
  * every merge is a left join onto a VRscores grain and keeps a match indicator

Run modules in numeric order; each writes into DATA and prints its diagnostics.
"""
from __future__ import annotations

import json
import os
import re
import sys
import time
import zipfile
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

try:
    import polars as pl
    HAVE_POLARS = True
except ImportError:
    HAVE_POLARS = False

# --------------------------------------------------------------------------- config
ROOT = Path(os.environ.get("POLISY_ROOT", "/content/polisy"))
CONFIG = {
    "RAW": ROOT / "data" / "raw",              # downloads land here, never edited
    "CANONICAL": ROOT / "data" / "canonical",  # one parquet per source, one grain each
    "KEYS": ROOT / "data" / "keys",            # crosswalks
    "PANELS": ROOT / "data" / "panels",        # analysis-ready panels
    "OUT": ROOT / "output",                    # tables and figures

    # VRscores inputs, as downloaded
    "VR_EMPLOYER": "/content/dataverse_files.zip",
    "VR_MSA": "/content/dataverse_files__msa.zip",
    "VR_INDUSTRY": "/content/dataverse_files_industries.zip",
    "VR_OCCUPATION": "/content/dataverse_files_Occupation.zip",

    # thesis files
    "COMPUSTAT": "/content/Compustat_Final.csv",
    "DIPI": "/content/Organizational_Leadership_File.csv",
    "COUNTYPRES": "/content/countypres_2000-2024.csv",

    # external
    "OEWS_YEARS": [2024],
    "CENSUS_API_KEY": "",
    "USER_AGENT": "academic research (University of Groningen)",
    "CBSA_REFERENCE": "/content/list1_2023.xlsx",
}
GRAINS = {
    "vr_employer": ("vrid", "year"),
    "vr_metro": ("msa", "year"),
    "vr_industry": ("naics6", "year"),
    "vr_occupation": ("occ_code", "year"),
    "oews_national": ("occ_code", "year"),
    "oews_metro": ("cbsa", "occ_code", "year"),
    "oews_industry": ("naics4", "occ_code", "year"),
    "acs_metro": ("cbsa", "year"),
    "vote_county": ("county_fips", "year"),
    "compustat": ("gvkey", "fyear"),
    "dipi": ("gvkey", "year"),
}

PRIMARY_STATES = {"GA", "IL", "IN", "MI", "MS", "OH", "SC", "TN", "TX", "VA", "WA"}
MODELED_STATES = {"AL", "HI", "MN", "MO", "MT", "ND", "WI", "VT"}
INVENTIVE_SOC2 = {"15": "Computer & mathematical", "17": "Architecture & engineering",
                  "19": "Life, physical & social science"}


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def paths():
    for k in ("RAW", "CANONICAL", "KEYS", "PANELS", "OUT"):
        Path(CONFIG[k]).mkdir(parents=True, exist_ok=True)
    (Path(CONFIG["OUT"]) / "tables").mkdir(exist_ok=True)
    (Path(CONFIG["OUT"]) / "figures").mkdir(exist_ok=True)
    return {k: Path(CONFIG[k]) for k in ("RAW", "CANONICAL", "KEYS", "PANELS", "OUT")}


def con():
    """One DuckDB connection with a spill directory, so 6M-row scans never OOM."""
    c = duckdb.connect()
    tmp = Path(CONFIG["CANONICAL"]).parent / "_duckdb_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    c.execute(f"SET temp_directory = '{tmp.as_posix()}'")
    c.execute("SET preserve_insertion_order = false")
    return c


def sqlp(p):
    return str(Path(p).as_posix()).replace("'", "''")


def q(c, sql):
    return c.execute(sql).df()


def q1(c, sql):
    r = c.execute(sql).fetchone()
    return None if r is None else r[0]


def save(df, name, note=""):
    """Every table the pipeline produces lands in output/tables as CSV."""
    p = Path(CONFIG["OUT"]) / "tables" / f"{name}.csv"
    df.to_csv(p, index=False)
    log(f"table {name}.csv ({len(df):,} rows){' - ' + note if note else ''}")
    return df


# ------------------------------------------------------------------- reading raw VRscores
def sniff_delim(path):
    """VRscores ships .tab files that are comma separated; never trust the extension."""
    with open(path, "rb") as fh:
        head = fh.readline(200000).decode("utf-8", "ignore")
    counts = {",": head.count(","), "\t": head.count("\t"), ";": head.count(";")}
    return max(counts, key=counts.get)


YEAR_RE = re.compile(r"(?<!\d)(20[0-4]\d)(?!\d)")


def zip_to_parquet(c, zip_path, out_path, add_year_from_name=True):
    """Unpack one member at a time, convert with DuckDB, delete the unpacked copy.

    Why: the employer ZIP holds 13 files of up to 175 MB uncompressed. This keeps peak
    disk use at one file and peak memory near zero.
    """
    out_path = Path(out_path)
    if out_path.exists():
        return out_path
    tmp = out_path.parent / "_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    parts = []
    with zipfile.ZipFile(zip_path) as zf:
        members = [n for n in zf.namelist()
                   if n.lower().endswith((".csv", ".tab")) and "__MACOSX" not in n]
        for name in sorted(members):
            dest = tmp / Path(name).name
            with zf.open(name) as fin, open(dest, "wb") as fout:
                fout.write(fin.read())
            delim = sniff_delim(dest)
            reader = (f"read_csv('{sqlp(dest)}', header=true, auto_detect=true, "
                      f"sample_size=-1, delim='{delim}')")
            cols = [x.lower() for x in q(c, f"DESCRIBE SELECT * FROM {reader}")["column_name"]]
            m = YEAR_RE.search(Path(name).name)
            add = f", {int(m.group(1))} AS year" if (add_year_from_name and "year" not in cols and m) else ""
            part = tmp / (Path(name).stem + ".parquet")
            c.execute(f"COPY (SELECT *{add} FROM {reader}) TO '{sqlp(part)}' (FORMAT parquet, COMPRESSION zstd)")
            dest.unlink()
            parts.append(part)
    lst = "[" + ", ".join(f"'{sqlp(p)}'" for p in parts) + "]"
    c.execute(f"COPY (SELECT * FROM read_parquet({lst}, union_by_name=true)) "
              f"TO '{sqlp(out_path)}' (FORMAT parquet, COMPRESSION zstd)")
    for p in parts:
        p.unlink()
    log(f"canonical {out_path.name}: {q1(c, f'SELECT count(*) FROM read_parquet(\'{sqlp(out_path)}\')'):,} rows")
    return out_path


def vr_view(c, name, parquet, kind):
    """Register a VRscores panel under canonical column names."""
    key = {"employer": "vrid", "metro": "msa", "industry": "naics_code", "occupation": "onet_code"}[kind]
    extra = "company_name, employee_count AS workers, avg_match_quality AS match_q," if kind == "employer" else ""
    c.execute(f"""CREATE OR REPLACE VIEW {name} AS SELECT
        CAST({key} AS VARCHAR) AS unit, CAST(year AS INTEGER) AS year, {extra}
        CAST(dem_workers_raw AS DOUBLE) AS dem_raw, CAST(rep_workers_raw AS DOUBLE) AS rep_raw,
        CAST(dem_workers_imp AS DOUBLE) AS dem, CAST(rep_workers_imp AS DOUBLE) AS rep,
        CAST(dem_workers_imp + rep_workers_imp AS DOUBLE) AS tp,
        rep_workers_imp / NULLIF(dem_workers_imp + rep_workers_imp, 0) AS rep_share,
        rep_workers_raw / NULLIF(dem_workers_raw + rep_workers_raw, 0) AS rep_share_raw
        FROM read_parquet('{sqlp(parquet)}')""")
    return name


# ------------------------------------------------------------------- canonical keys
def norm_city(s):
    s = str(s).lower().replace("st.", "saint").replace("ste.", "sainte")
    return re.sub(r"[^a-z ]", "", s).strip()


def parse_msa(name):
    """'Boston-Cambridge-Quincy MA-NH MSA' -> (cities, states)."""
    s = str(name).strip()
    m = re.search(r"\s([A-Z]{2}(?:-[A-Z]{2})*)(?:\s+(?:MSA|Metro Area|Micro Area))?\s*$", s)
    states = m.group(1).split("-") if m else []
    base = s[:m.start()] if m else s
    return [norm_city(c) for c in re.split(r"[-/]", base) if c.strip()], states


def party_regime(state):
    if not state:
        return "unknown"
    if state in PRIMARY_STATES:
        return "primary-based"
    if state in MODELED_STATES:
        return "modelled by L2"
    return "party registration"


LEGAL_TAIL = {"inc", "incorporated", "corp", "corporation", "co", "company", "companies", "llc",
              "ltd", "limited", "plc", "lp", "llp", "holdings", "holding", "group", "the", "and"}
ABBR = {"international": "intl", "technologies": "tech", "technology": "tech", "systems": "sys",
        "services": "svcs", "communications": "comm", "manufacturing": "mfg",
        "pharmaceuticals": "pharma", "laboratories": "labs", "industries": "inds",
        "financial": "finl", "national": "natl", "american": "amer", "america": "amer",
        "corporation": "corp", "company": "co", "incorporated": "inc"}


def norm_name(s):
    """Canonical company name, used on both sides of every name match."""
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\(.*?\)", " ", s.lower())
    s = re.sub(r"/[a-z]{2,4}/?", " ", s)
    s = re.sub(r"-\s*(cl|class)\s*[a-z]\b", " ", s)
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    toks = [ABBR.get(t, t) for t in s.split()]
    out, run = [], []
    for t in toks:                                  # "u s steel" -> "us steel"
        if len(t) == 1 and t.isalpha():
            run.append(t)
            continue
        if run:
            out.append("".join(run)); run = []
        out.append(t)
    if run:
        out.append("".join(run))
    while out and out[0] == "the":
        out = out[1:]
    while out and out[-1] in LEGAL_TAIL:
        out.pop()
    return " ".join(out)


# ------------------------------------------------------------------- shared analysis
def exposure_by_party(dem, rep):
    """Leave-one-out own-party coworker exposure divided by the national party share.

    Returns (democratic ratio, republican ratio, combined). 1.00 = random mixing. The
    leave-one-out form makes the benchmark exactly 1 for units of any size, which is why
    small employers do not bias it.
    """
    d, r = np.round(np.asarray(dem, float)), np.round(np.asarray(rep, float))
    t = d + r
    keep = t >= 2
    d, r, t = d[keep], r[keep], t[keep]
    D, R, T = d.sum(), r.sum(), t.sum()
    if D == 0 or R == 0:
        return np.nan, np.nan, np.nan
    ed = (d * (d - 1) / (t - 1)).sum() / D
    er = (r * (r - 1) / (t - 1)).sum() / R
    return ed / (D / T), er / (R / T), (D * ed / (D / T) + R * er / (R / T)) / T


def weighted_corr(x, y, w):
    x, y, w = np.asarray(x, float), np.asarray(y, float), np.asarray(w, float)
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    cov = np.average((x - mx) * (y - my), weights=w)
    return cov / np.sqrt(np.average((x - mx) ** 2, weights=w) * np.average((y - my) ** 2, weights=w))


def coverage_note(matched_workers, total_workers):
    """Every merged result carries this line. Non-negotiable."""
    return f"covers {matched_workers / total_workers:.1%} of VRscores matched workers"

The `polisy_core.py` file has been updated. Now, please re-run cell `4b743595` to check the data inventory with the corrected configuration.

The `polisy_core.py` file has been created. Now, I will run the `main()` function from the data inventory cell.

In [43]:
import importlib
from pathlib import Path
import zipfile
import pandas as pd

# 1. Reload the already-saved polisy_core module
import polisy_core
importlib.reload(polisy_core)

# 2. Run the inventory pipeline cleanly
CODEBOOK_ROWS = {2012: 370426, 2013: 393571, 2014: 417604, 2015: 439982, 2016: 461191,
                 2017: 482144, 2018: 502104, 2019: 516553, 2020: 525013, 2021: 535238,
                 2022: 539944, 2023: 540676, 2024: 534392}

def main():
    polisy_core.paths()
    rows = []
    for key in ("VR_EMPLOYER", "VR_MSA", "VR_INDUSTRY", "VR_OCCUPATION", "COMPUSTAT", "DIPI", "COUNTYPRES", "CBSA_REFERENCE"):
        p = Path(polisy_core.CONFIG[key])
        if not p.exists():
            rows.append({"asset": key, "path": str(p), "status": "MISSING"})
            continue
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p) as zf:
                members = [i for i in zf.infolist() if not i.is_dir()]
                for i in members:
                    rows.append({"asset": key, "path": i.filename, "status": "ok",
                                 "format": Path(i.filename).suffix.lstrip("."),
                                 "mb_compressed": round(i.compress_size / 1e6, 2),
                                 "mb_raw": round(i.file_size / 1e6, 2)})
        else:
            rows.append({"asset": key, "path": p.name, "status": "ok",
                         "format": p.suffix.lstrip("."), "mb_raw": round(p.stat().st_size / 1e6, 2)})
    inv = pd.DataFrame(rows)
    polisy_core.save(inv, "01_inventory")
    missing = inv[inv.status == "MISSING"]
    if len(missing):
        polisy_core.log(f"MISSING assets: {', '.join(missing.asset)}")
    polisy_core.log(f"{(inv.status == 'ok').sum()} files found, {inv.get('mb_raw', pd.Series(dtype=float)).sum():,.0f} MB raw")
    return inv

main()

[16:18:15] table 01_inventory.csv (60 rows)
[16:18:15] MISSING assets: DIPI, CBSA_REFERENCE
[16:18:15] 58 files found, 2,420 MB raw


,asset,path,status,format,mb_compressed,mb_raw
0,VR_EMPLOYER,codebook.md,ok,md,0.00,0.01
1,VR_EMPLOYER,employer_panel_year_2012.tab,ok,tab,36.66,137.24
2,VR_EMPLOYER,employer_panel_year_2013.tab,ok,tab,38.98,145.81
3,VR_EMPLOYER,employer_panel_year_2014.tab,ok,tab,41.39,154.67
4,VR_EMPLOYER,employer_panel_year_2015.csv,ok,csv,43.62,162.91
5,VR_EMPLOYER,employer_panel_year_2016.csv,ok,csv,45.72,170.72
6,VR_EMPLOYER,employer_panel_year_2017.csv,ok,csv,47.80,178.39
7,VR_EMPLOYER,employer_panel_year_2018.csv,ok,csv,49.81,185.73
8,VR_EMPLOYER,employer_panel_year_2019.csv,ok,csv,51.27,191.08
9,VR_EMPLOYER,employer_panel_year_2020.csv,ok,csv,51.97,193.96


In [13]:
# -*- coding: utf-8 -*-
"""02 VRscores cleaning: raw ZIPs to canonical Parquet.

What it does: converts the four VRscores panels to one Parquet each, adding the year
from the file name where the panel does not carry it (industry and occupation).
Why: the employer panel is 6.26m rows; Parquet plus DuckDB makes every later query
seconds rather than minutes, and pandas never sees the whole thing.
Expect: data/canonical/vr_employer.parquet (about 450 MB), vr_metro, vr_industry,
vr_occupation, and output/tables/02_vr_year_counts.csv.
Diagnostics: year counts against the codebook; duplicate unit-year rows must be zero;
occupation codes that are not SOC-shaped ("Retired", "unknown", "On Leave") are reported
and dropped downstream, not here.
"""
import pandas as pd
from pathlib import Path
from polisy_core import CONFIG, paths, con, log, save, zip_to_parquet, vr_view, q, sqlp

PANELS = {"employer": "VR_EMPLOYER", "metro": "VR_MSA",
          "industry": "VR_INDUSTRY", "occupation": "VR_OCCUPATION"}


def main():
    P = paths()
    c = con()
    out = {}
    for kind, key in PANELS.items():
        src = CONFIG[key]
        if not str(src) or not Path(str(src)).exists():
            log(f"{kind}: source missing, skipped")
            continue
        pq = zip_to_parquet(c, src, P["CANONICAL"] / f"vr_{kind}.parquet")
        out[kind] = pq
        vr_view(c, f"vr_{kind}", pq, kind)
        dup = q(c, f"SELECT count(*) - count(DISTINCT (unit, year)) AS dups FROM vr_{kind}").iloc[0, 0]
        yrs = q(c, f"SELECT year, count(*) AS rows, sum(tp) AS two_party_workers FROM vr_{kind} GROUP BY 1 ORDER BY 1")
        yrs.insert(0, "panel", kind)
        save(yrs, f"02_year_counts_{kind}", f"{dup} duplicate unit-years")
        if dup:
            log(f"WARNING {kind}: {dup} duplicate unit-year rows")
    return out


if __name__ == "__main__":
    main()

[15:55:15] table 02_year_counts_employer.csv (13 rows) - 0 duplicate unit-years
[15:55:16] canonical vr_metro.parquet: 4,758 rows
[15:55:16] table 02_year_counts_metro.csv (13 rows) - 0 duplicate unit-years
[15:55:16] table 02_year_counts_industry.csv (13 rows) - 0 duplicate unit-years
[15:55:16] table 02_year_counts_occupation.csv (13 rows) - 0 duplicate unit-years


In [14]:
# -*- coding: utf-8 -*-
"""03 BLS OEWS cleaning.

What it does: reads the OEWS national, metro and industry zips, keeps detailed
occupations only, and writes three canonical Parquet tables.
Why: OEWS stacks totals, major, minor, broad and detailed rows in one column. Summing
without filtering O_GROUP == 'detailed' double counts employment by roughly a factor of
three. Metro rows are AREA_TYPE 4, not 2.
Expect: canonical/oews_national.parquet (about 830 occupations), oews_metro.parquet
(about 141k rows), oews_industry.parquet (4-digit NAICS, about 83k rows).
Diagnostics: total detailed employment should be close to the published US total
(about 151m in May 2024); metro count should be near 390.
"""
import zipfile
from pathlib import Path
import pandas as pd
from polisy_core import CONFIG, paths, log, save

LEVELS = {"national": ("nat", "national_m"), "msa": ("ma", "msa_m"), "industry": ("in4", "nat4d_m")}


def read_level(raw: Path, year: int, level: str):
    suffix, prefer = LEVELS[level]
    zpath = raw / f"oesm{str(year)[2:]}{suffix}.zip"
    if not zpath.exists():
        log(f"OEWS {level} {year}: {zpath.name} not found in {raw}")
        return None
    with zipfile.ZipFile(zpath) as zf:
        sheets = [n for n in zf.namelist() if n.lower().endswith((".xlsx", ".xls"))
                  and "file_description" not in n.lower()]
        member = next((n for n in sheets if prefer in Path(n).name.lower()), sheets[0])
        df = pd.read_excel(zf.open(member), dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    for col in ("tot_emp", "a_median", "a_mean"):
        if col in df:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
    if "o_group" in df:
        df = df[df.o_group.str.lower() == "detailed"]
    if level == "msa" and "area_type" in df:
        df = df[df.area_type.astype(str).str.strip() == "4"]
    if level == "industry" and "i_group" in df:
        df = df[df.i_group.str.contains("digit", case=False, na=False)]
    keep = [c for c in ["area", "area_title", "naics", "naics_title", "occ_code", "occ_title",
                        "tot_emp", "a_median"] if c in df]
    out = df[keep].copy()
    out["year"] = year
    if level == "msa":
        out = out.rename(columns={"area": "cbsa"})
    if level == "industry":
        out["naics4"] = out["naics"].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:4]
    log(f"OEWS {level} {year}: {len(out):,} detailed rows")
    return out


def main():
    P = paths()
    for level in LEVELS:
        frames = [read_level(P["RAW"], y, level) for y in CONFIG["OEWS_YEARS"]]
        frames = [f for f in frames if f is not None]
        if not frames:
            continue
        df = pd.concat(frames, ignore_index=True)
        df.to_parquet(P["CANONICAL"] / f"oews_{level}.parquet", index=False)
        if level == "national":
            save(pd.DataFrame([{"year": y, "occupations": (df.year == y).sum(),
                                "total_employment": df.loc[df.year == y, "tot_emp"].sum()}
                               for y in sorted(df.year.unique())]), "03_oews_national_check",
                 "compare total employment with the published US total")
        if level == "msa":
            log(f"metros in OEWS: {df.cbsa.nunique()}")
    return True


if __name__ == "__main__":
    main()

[15:56:22] OEWS national 2024: oesm24nat.zip not found in /content/polisy/data/raw
[15:56:22] OEWS msa 2024: oesm24ma.zip not found in /content/polisy/data/raw
[15:56:22] OEWS industry 2024: oesm24in4.zip not found in /content/polisy/data/raw


In [15]:
# -*- coding: utf-8 -*-
"""04 Identifier crosswalks: metro, occupation, industry, and the firm name match.

What it does: builds the four key tables every later merge uses.
  keys/cw_msa_cbsa.csv     VRscores metro name  -> CBSA code (+ state, party regime)
  keys/cw_occupation.csv   VRscores O*NET code  -> SOC6 (invalid codes flagged)
  keys/cw_industry.csv     VRscores NAICS6      -> NAICS4 (OEWS resolution)
  keys/cw_vrid_gvkey.csv   VRscores employer    -> gvkey, by name with evidence
Why: merges fail silently when keys are built inline. Building them once, with a match
score per row, makes every later join auditable.
Expect: match rates near 97% for metros, 100% for occupations after dropping invalid
codes, 71% for industries, and roughly 35% of Compustat firms for the name match.
Diagnostics: review every metro row with match_score < 3 by hand once; check the firm
match against DIPI (module 07).
"""
import re
from pathlib import Path
import numpy as np
import pandas as pd
from polisy_core import (CONFIG, paths, con, log, save, q, sqlp, parse_msa, party_regime,
                         norm_name, vr_view)

try:
    from rapidfuzz import fuzz, process
    HAVE_RF = True
except ImportError:
    HAVE_RF = False


def cbsa_reference(P):
    """Census delineation file if present, else the OEWS metro file (codes + titles)."""
    delin = P["RAW"] / "list1_2023.xlsx"
    if delin.exists():
        raw = pd.read_excel(delin, skiprows=2, dtype=str)
        raw.columns = [re.sub(r"\s+", "_", c.strip().lower()) for c in raw.columns]
        code = next(c for c in raw.columns if c.startswith("cbsa_code"))
        title = next(c for c in raw.columns if c.startswith("cbsa_title"))
        typ = next((c for c in raw.columns if "metropolitan" in c and "division" not in c), None)
        st = next((c for c in raw.columns if c.startswith("fips_state")), None)
        cty = next((c for c in raw.columns if c.startswith("fips_county")), None)
        d = raw.rename(columns={code: "cbsa", title: "cbsa_title", typ: "cbsa_type",
                                st: "state_fips", cty: "county_fips3"})
        d = d[d.cbsa.str.match(r"^\d{5}$", na=False)].copy()
        if "state_fips" in d and "county_fips3" in d:
            d["county_fips"] = d.state_fips.str.zfill(2) + d.county_fips3.str.zfill(3)
            d[["cbsa", "county_fips"]].dropna().to_csv(P["KEYS"] / "cw_cbsa_county.csv", index=False)
        log("CBSA reference: Census delineation file")
        return d[["cbsa", "cbsa_title"]].drop_duplicates()
    oews = P["CANONICAL"] / "oews_msa.parquet"
    if oews.exists():
        d = pd.read_parquet(oews, columns=["cbsa", "area_title"]).drop_duplicates()
        log("CBSA reference: OEWS metro file (no county links; election merges need the Census file)")
        return d.rename(columns={"area_title": "cbsa_title"})
    log("no CBSA reference available")
    return None


def build_metro_crosswalk(P, vr_msa_names, ref):
    parsed = ref.cbsa_title.map(parse_msa)
    ref = ref.assign(cities=[p[0] for p in parsed], states=[p[1] for p in parsed])
    by_state = {}
    for r in ref.itertuples(index=False):
        for s in r.states:
            by_state.setdefault(s, []).append(r)
    rows = []
    for name in pd.Series(vr_msa_names).dropna().unique():
        cities, states = parse_msa(name)
        best, score = None, 0
        for cand in by_state.get(states[0] if states else "", []):
            s = 3 if cities and cand.cities and cities[0] == cand.cities[0] else (2 if set(cities) & set(cand.cities) else 0)
            if s > score:
                best, score = cand, s
        rows.append({"msa": name, "cbsa": best.cbsa if best is not None else None,
                     "cbsa_title": best.cbsa_title if best is not None else None,
                     "match_score": score, "state": states[0] if states else None,
                     "party_regime": party_regime(states[0] if states else None)})
    cw = pd.DataFrame(rows)
    cw.to_csv(P["KEYS"] / "cw_msa_cbsa.csv", index=False)
    log(f"metro crosswalk: {cw.cbsa.notna().mean():.1%} matched, {(cw.match_score == 3).mean():.1%} on the principal city")
    return cw


def build_firm_crosswalk(P, c, year, min_workers=25, min_score=90):
    """Name match VRscores employers to Compustat gvkeys, with size and timing evidence."""
    comp = pd.read_csv(CONFIG["COMPUSTAT"], usecols=["gvkey", "fyear", "conm", "state", "emp"], low_memory=False)
    comp = comp[comp.fyear == year].dropna(subset=["conm"]).drop_duplicates("gvkey")
    comp["gvkey"] = comp.gvkey.astype(str).str.zfill(6)
    comp["name_norm"] = comp.conm.map(norm_name)
    emp = q(c, f"""SELECT unit AS vrid, company_name, workers, tp FROM vr_employer
                   WHERE year = {year} AND workers >= {min_workers}""")
    emp["name_norm"] = emp.company_name.map(norm_name)
    emp = emp[emp.name_norm != ""]
    exact = comp.merge(emp, on="name_norm", how="inner").assign(method="exact", score=100.0)
    rows = [exact]
    if HAVE_RF:
        pool = emp.name_norm.to_numpy()
        blocks = {}
        for i, n in enumerate(pool):
            blocks.setdefault(n.split()[0], []).append(i)
        fuzzy = []
        unmatched = comp[~comp.gvkey.isin(exact.gvkey)]
        for f in unmatched.itertuples(index=False):
            idx = blocks.get(f.name_norm.split()[0] if f.name_norm else "", [])
            if not idx:
                continue
            hits = process.extract(f.name_norm, pool[idx], scorer=fuzz.token_sort_ratio,
                                   limit=3, score_cutoff=min_score)
            for _, sc, j in hits:
                k = idx[j]
                fuzzy.append({"gvkey": f.gvkey, "conm": f.conm, "state": f.state, "emp": f.emp,
                              "name_norm": f.name_norm, "vrid": emp.vrid.iloc[k],
                              "company_name": emp.company_name.iloc[k], "workers": emp.workers.iloc[k],
                              "tp": emp.tp.iloc[k], "method": "fuzzy", "score": float(sc)})
        if fuzzy:
            rows.append(pd.DataFrame(fuzzy))
    cand = pd.concat(rows, ignore_index=True)
    emp_total = q(c, f"SELECT sum(workers) FROM vr_employer WHERE year = {year}").iloc[0, 0]
    cand["size_ratio"] = cand.workers / (pd.to_numeric(cand.emp, errors="coerce") * 1000)
    cand["size_ok"] = cand.size_ratio.between(0.002, 1.5)
    cand["tier"] = np.where((cand.method == "exact") & cand.size_ok.fillna(False), "high",
                   np.where(cand.method == "exact", "medium",
                   np.where(cand.size_ok.fillna(False) & (cand.score >= 95), "medium", "low")))
    cand["accept"] = pd.NA          # fill by hand for anything below high
    cand.to_csv(P["KEYS"] / "cw_vrid_gvkey.csv", index=False)
    high = cand[cand.tier == "high"]
    log(f"firm crosswalk {year}: {cand.gvkey.nunique():,} Compustat firms with a candidate, "
        f"{high.gvkey.nunique():,} at high confidence, covering {high.workers.sum() / emp_total:.1%} of VRscores workers")
    return cand


def main(year=2015):
    P = paths()
    c = con()
    for kind in ("employer", "metro", "industry", "occupation"):
        pq = P["CANONICAL"] / f"vr_{kind}.parquet"
        if pq.exists():
            vr_view(c, f"vr_{kind}", pq, kind)
    ref = cbsa_reference(P)
    if ref is not None and (P["CANONICAL"] / "vr_metro.parquet").exists():
        names = q(c, "SELECT DISTINCT unit AS msa FROM vr_metro").msa
        build_metro_crosswalk(P, names, ref)
    if (P["CANONICAL"] / "vr_occupation.parquet").exists():
        occ = q(c, "SELECT DISTINCT unit AS onet_code FROM vr_occupation")
        occ["valid"] = occ.onet_code.str.match(r"^\d{2}-\d{4}")
        occ["occ_code"] = occ.onet_code.str[:7].where(occ.valid)
        occ.to_csv(P["KEYS"] / "cw_occupation.csv", index=False)
        log(f"occupation crosswalk: {occ.valid.mean():.1%} valid SOC codes, dropping {(~occ.valid).sum()} non-occupations")
    if (P["CANONICAL"] / "vr_industry.parquet").exists():
        ind = q(c, "SELECT DISTINCT unit AS naics6 FROM vr_industry")
        ind["naics4"] = ind.naics6.str.zfill(6).str[:4]
        ind.to_csv(P["KEYS"] / "cw_industry.csv", index=False)
        log(f"industry crosswalk: {ind.naics4.nunique()} four-digit industries")
    if Path(CONFIG["COMPUSTAT"]).exists() and (P["CANONICAL"] / "vr_employer.parquet").exists():
        build_firm_crosswalk(P, c, year)
    return True


if __name__ == "__main__":
    main()

[15:56:40] no CBSA reference available
[15:56:40] occupation crosswalk: 99.7% valid SOC codes, dropping 3 non-occupations
[15:56:40] industry crosswalk: 309 four-digit industries
[15:56:43] firm crosswalk 2015: 2,708 Compustat firms with a candidate, 2,500 at high confidence, covering 16.8% of VRscores workers
